***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import itertools
# pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.max_columns', None)

from __future__ import annotations
from typing import Union
import urllib3
import json
from distutils.log import warn


## Setting file paths ---

# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'HUD')

path_code    = os.path.join(path_git, 'Data', 'HUD')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')


## Setting API key ---

# Obtain API Key from the following source 
# https://www.huduser.gov/portal/dataset/fmr-api.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()



***

Importing

***

In [ ]:
http = urllib3.PoolManager()

headers = {
        "Authorization": "Bearer " + api_key,
        "User-Agent": "https://github.com/etam4260/hudpy"
    }

In [ ]:
url = "https://www.huduser.gov/hudapi/public/chas?type=2&year=2017-2021&stateId=6"  # Example for California, 2023
response = requests.get(url, headers=headers)
data = json.loads(response.text)
df = pd.DataFrame(data[0], index=[0])
print(df.shape)
display(df.head());print('')
df_cols = pd.read_excel(os.path.join(path_config, 'API to manual download mapping.xlsx'), sheet_name='API_CHAS_DataDictionary')
df_cols = df_cols[['property', 'description']]
display(df_cols.head())
dict_cols = df_cols.set_index('property').to_dict()['description']
df = df.rename(columns=dict_cols)
df

In [ ]:

df_codes = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name='CDPcodes')
df_codes = df_codes[df_codes['MPO'] == 'SACOG']
CDPcodes = list(df_codes['place'].unique())

list_df = []

for place in tqdm(CDPcodes):
    try:
        url = f"https://www.huduser.gov/hudapi/public/chas?type=5&year=2017-2021&stateId=6&entityId={place}"  # Example for California, 2023
        response = requests.get(url, headers=headers)
        data = json.loads(response.text)
        df = pd.DataFrame(data[0], index=[0])
        df_cols = pd.read_excel(os.path.join(path_config, 'API to manual download mapping.xlsx'), sheet_name='API_CHAS_DataDictionary')
        df_cols = df_cols[['property', 'description']]
        dict_cols = df_cols.set_index('property').to_dict()['description']
        df = df.rename(columns=dict_cols)
        list_df.append(df)
        time.sleep(3)
    except Exception as e: print(e)

df = pd.concat(list_df)
display(df.head(), df.tail())

In [ ]:
df = pd.concat(list_df)
display(df.head(), df.tail())

Code graveyard

In [ ]:
# url = "https://www.huduser.gov/hudapi/public/chas/listCities/6"  # Example for California, 2023
# response = requests.get(url, headers=headers)
# data = json.loads(response.text)
# data

In [ ]:
# # https://github.com/etam4260/hudpy/blob/main/src/hudpy/hud_chas.py
# import hudpy
# print(dir(hudpy))
# from hudpy import hud_internet_online
# from hudpy import hud_input_check
# from hudpy import hud_do_query_calls
# from hudpy import hud_pkg_env
# from hudpy import hud_misc
# from hudpy import hud_pkg_env, hud_download_bar


# def hud_set_key(key:str):
#     """
#     Function to set the HUD_KEY environment variable.

#     Parameters
#     ----------

#     key : The key obtained at https://www.huduser.gov/hudapi/public/register?comingfrom=1.

#     See Also
#     --------
#     * hud_get_key()

#     Examples
#     --------
#     >>> hud_set_key("DWKQOD442OLKDF3")
#     """
#     if not isinstance(key, str):
#         raise ValueError("Key should be a string.")

#     os.environ["HUD_KEY"] = key

# hud_set_key(api_key)




# def hud_chas_state(state: Union[int, str, list[int], list[str], tuple[int], tuple[str]],
#                    year: Union[int, str, list[int], list[str], tuple[int], tuple[str]] = "2014-2018", 
#                    key: str = None):
#     """
#     Function to query Comprehensive Housing and Affordability (CHAS) API provided
#     by the US Department of Housing and Urban Development. This returns CHAS measurements
#     for state(s)

#     Parameters
#     ----------

#     state : The state(s) to query for CHAS. Can be provided as the full name, fip code or
#         abbreviation.

#     year : The year(s) to query for.
#          * 2014-2018
#          * 2013-2017
#          * 2012-2016
#          * 2011-2015
#          * 2010-2014
#          * 2009-2013
#          * 2008-2012
#          * 2007-2011
#          * 2006-2010


#     key : The API key for this user. You must go to HUD and sign up for an
#         account and request for an API key.

#     See Also
#     --------
    
#     * hud_chas()
#     * hud_chas_nation()
#     * hud_chas_state()
#     * hud_chas_county()
#     * hud_chas_place()
#     * hud_chas_mcd()

#     Returns
#     -------
#     This returns CHAS data for state(s) query.

#     Examples
#     --------

#     >>> hud_chas_state("MD")
   
#     >>> hud_chas_state("24")

#     >>> hud_chas_state("Maryland")

#     """
    
#     if hud_pkg_env.pkg_env["internet_on"] == False: 
#         if not hud_internet_online.internet_on():
#             raise ConnectionError("You currently do not have internet access.")
#         else:
#             hud_pkg_env.pkg_env["internet_on"] == True
            
#     if(key == None and os.getenv("HUD_KEY") != None):
#         key = os.getenv("HUD_KEY")


#     args = hud_input_check.chas_input_check_cleansing(state, year, key)
#     state = args[0]
#     year = args[1]
#     key = args[2]
  
#     # Assume abbreviation or fips code if length of 2. Captitalize does not
#     # affect numbers. Assume full state name if length more than 2
#     if all(map(lambda x: len(x) == 2, state)):
#         state = list(map(lambda x: str.upper(x), state))
#     elif all(map(lambda x: len(x) > 2, state)):
#         state = list(map(lambda x: x[0:1].upper() + x[1:len(x)].lower(), state))

#     if hud_pkg_env.pkg_env["states"].empty:
#         hud_pkg_env.pkg_env["states"] = hud_misc.hud_nation_states_territories(key = key)
#         hud_pkg_env.pkg_env["states"]["state_num"] = hud_pkg_env.pkg_env["states"]["state_num"].astype("float").astype("int").astype("str")
        
#     for i in range(0, len(state)):
#         if state[i] not in hud_pkg_env.pkg_env["states"].values:
#             raise ValueError("There is no matching fips code for " + str(state[i]))

#     if len(set(state).intersection(set(hud_pkg_env.pkg_env["states"]["state_name"]))) != 0: 
#         # Not sure if this is right syntax... need to test it...
#         state = list(hud_pkg_env.pkg_env["states"][hud_pkg_env.pkg_env["states"]["state_name"].isin(state)]["state_num"])
#     if len(set(state).intersection(set(hud_pkg_env.pkg_env["states"]["state_code"]))) != 0:   
#         state = list(hud_pkg_env.pkg_env["states"][hud_pkg_env.pkg_env["states"]["state_code"].isin(state)]["state_num"]) 
#     if len(set(state).intersection(set(hud_pkg_env.pkg_env["states"]["state_num"]))) != 0:  
#         state = list(hud_pkg_env.pkg_env["states"][hud_pkg_env.pkg_env["states"]["state_num"].isin(state)]["state_num"])   
    
  
    
#     all_queries = list(itertools.product(["https://www.huduser.gov/hudapi/public/chas?type=2&stateId="], 
#                                          state, ["&year="], year))
    
#     urls = []
#     for i in range(len(all_queries)):
#         urls.append(
#             all_queries[i][0] + 
#             all_queries[i][1] +
#             all_queries[i][2] +
#             all_queries[i][3]
#         )
  

#     return hud_do_query_calls.chas_do_query_calls(urls, key = key)

# hud_chas_state("Maryland", year = ["2012-2016"])
